# Phase 2 — Transportation / Road Context

**BACKEND_PLAN.md §2** · The agent must know its surroundings — at minimum the biggest road near the site.

This notebook validates the `road_context.analyze_roads` tool by:
1. Setting up a complex splayed site with three roads of different widths and hierarchies.
2. Identifying and visualising the **main road** (highest hierarchy → widest → most frontage).
3. Showing **side tagging** — which site side each road fronts.
4. Deriving **per-edge setbacks** from road widths and visualising the resulting buildable zone.
5. Showing that the **site grid aligns to the main-road side** automatically (Phase 2 ↔ Phase 3 integration).
6. Demonstrating the **ambiguity path** when no road data is supplied.

No LLM, no MCP. All geometry is deterministic.


In [ ]:
import sys, os, math
from pathlib import Path

# Locate the workspace root (the directory that contains the 'team_04' package).
# Works whether Jupyter cwd is AIA26_Studio, team_04, or team_04/test_notebooks.
_cwd = Path(os.getcwd()).resolve()
_root = next(
    (p for p in [_cwd, _cwd.parent, _cwd.parent.parent] if (p / 'team_04').is_dir()),
    None,
)
if _root is None:
    raise FileNotFoundError(
        'Cannot locate workspace root (directory containing team_04/). '
        f'Tried: {_cwd}, {_cwd.parent}, {_cwd.parent.parent}'
    )
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from shapely.geometry import LineString, Polygon

from team_04.agent.tools.road_context import analyze_roads, HIERARCHY_RANK
from team_04.agent.tools.site_model import build_site_model
from team_04.agent.tools.site_grid import derive_site_grid
from team_04.agent.tools.site_setback import compute_buildable_zone

print('Imports OK')

In [ ]:
# ---------------------------------------------------------------------------
# Scene definition
# ---------------------------------------------------------------------------

# A splayed non-orthogonal pentagon (same as grid/sun notebooks).
SITE = [[0, 0, 0], [130, 18, 0], [150, 92, 0], [62, 128, 0], [-14, 74, 0], [0, 0, 0]]

# Three roads of different hierarchies / widths.
MAIN_ROAD = {
    'type': 'road',
    'centerline': [[-20, -15], [160, 25]],
    'width_m': 20.0,
    'hierarchy': 'main',
    'name': 'Main Street',
}
SECONDARY_ROAD = {
    'type': 'road',
    'centerline': [[170, 0], [175, 110]],
    'width_m': 10.0,
    'hierarchy': 'secondary',
    'name': 'East Lane',
}
PATH_ROAD = {
    'type': 'road',
    'centerline': [[-20, 90], [75, 145]],
    'width_m': 4.0,
    'hierarchy': 'path',
    'name': 'West Path',
}

# Build the full site model (roads flow in via site_objects).
payload = {
    'site_objects': [MAIN_ROAD, SECONDARY_ROAD, PATH_ROAD],
    'default_setback': 5.0,
}
model = build_site_model(SITE, payload)
roads_result = model['roads']

print('Site model available:', model['available'])
print('Road analysis available:', roads_result['available'])
print('Number of roads analysed:', len(roads_result['roads']))
print('Main road:', roads_result['main_road']['name'],
      '|', roads_result['main_road']['hierarchy'],
      '| width', roads_result['main_road']['width_m'], 'm',
      '| nearest side', roads_result['main_road_side_index'])
print('edge_road_widths:', roads_result['edge_road_widths'])

In [ ]:
# ---------------------------------------------------------------------------
# §1 — Road identification: site + 3 roads coloured by hierarchy
# ---------------------------------------------------------------------------

HIER_COLOUR = {'main': '#E63946', 'secondary': '#F4A261', 'path': '#2A9D8F'}
HIER_LABEL = {'main': 'Main road (hierarchy=main, 20 m wide)',
               'secondary': 'Secondary road (10 m wide)',
               'path': 'Path (4 m wide)'}

site_pts = [(p[0], p[1]) for p in SITE]
site_poly = Polygon(site_pts)

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_aspect('equal')
ax.set_title('§1  Road identification — main road highlighted', fontsize=12, fontweight='bold')

# Site polygon
xs, ys = zip(*site_pts)
ax.fill(xs, ys, alpha=0.12, color='#457B9D')
ax.plot(xs, ys, '-', color='#1D3557', lw=2, label='Site boundary')

# Roads
for road in [MAIN_ROAD, SECONDARY_ROAD, PATH_ROAD]:
    cl = road['centerline']
    lw = road['width_m'] / 4.0
    clr = HIER_COLOUR[road['hierarchy']]
    xs_r = [p[0] for p in cl]
    ys_r = [p[1] for p in cl]
    ax.plot(xs_r, ys_r, '-', color=clr, lw=max(lw, 1.5),
            label=HIER_LABEL[road['hierarchy']])
    # Road width buffer (visual)
    buf = LineString(cl).buffer(road['width_m'] / 2.0)
    bx, by = buf.exterior.xy
    ax.fill(bx, by, alpha=0.12, color=clr)

# Highlight main road side on the site
main_si = roads_result['main_road_side_index']
sides = model['sides']
s = sides[main_si]
nodes = model['corners']
p1 = nodes[s['from_node_index']]['point']
p2 = nodes[s['to_node_index']]['point']
ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '-', color='#E63946', lw=5, alpha=0.5,
        label=f'Main-road side (side {main_si})')

# Label roads
for road in roads_result['roads']:
    cl = road['centerline']
    mx = (cl[0][0] + cl[-1][0]) / 2
    my = (cl[0][1] + cl[-1][1]) / 2
    ax.annotate(
        f"{road['name']}\n{road['hierarchy']}, {road['width_m']} m",
        xy=(mx, my), fontsize=8, ha='center', va='center',
        color=HIER_COLOUR[road['hierarchy']],
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7, ec='none'),
    )

ax.legend(fontsize=8, loc='upper right')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
plt.tight_layout()
plt.savefig('road_context_1_identification.png', dpi=100)
plt.show()
print('Main road =', roads_result['main_road']['name'],
      '| fronts side', roads_result['main_road_side_index'])

In [ ]:
# ---------------------------------------------------------------------------
# §2 — Side tagging: which site side does each road front?
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_aspect('equal')
ax.set_title('§2  Side tagging — adjacent_road on each site edge', fontsize=12, fontweight='bold')

# Site polygon (background)
sp = [(p[0], p[1]) for p in SITE]
xs, ys = zip(*sp)
ax.fill(xs, ys, alpha=0.08, color='#457B9D')
ax.plot(xs, ys, '-', color='#1D3557', lw=1.5)

# Draw each side coloured by its adjacent road
for side in model['sides']:
    si = side['edge_index']
    pA = nodes[side['from_node_index']]['point']
    pB = nodes[side['to_node_index']]['point']
    mx, my = (pA[0] + pB[0]) / 2, (pA[1] + pB[1]) / 2
    adj = side.get('adjacent_road')
    if adj:
        clr = HIER_COLOUR[adj['hierarchy']]
        lbl = f"side {si}: {adj['name']} ({adj['hierarchy']}, {adj['width_m']} m)"
    else:
        clr = '#AAAAAA'
        lbl = f"side {si}: no adjacent road"
    ax.plot([pA[0], pB[0]], [pA[1], pB[1]], '-', color=clr, lw=6, alpha=0.7)
    ax.annotate(lbl, xy=(mx, my), fontsize=7, ha='center', va='center',
                color=clr,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.85, ec='none'))

# Corners
for corner in nodes:
    ax.plot(corner['point'][0], corner['point'][1], 'o', color='#1D3557', ms=5)

# Roads (thin lines)
for road in [MAIN_ROAD, SECONDARY_ROAD, PATH_ROAD]:
    cl = road['centerline']
    ax.plot([p[0] for p in cl], [p[1] for p in cl],
            '--', color=HIER_COLOUR[road['hierarchy']], lw=1.5, alpha=0.6)

patches = [mpatches.Patch(color=HIER_COLOUR[h], label=h) for h in HIER_COLOUR]
patches.append(mpatches.Patch(color='#AAAAAA', label='no road'))
ax.legend(handles=patches, fontsize=8, loc='upper right')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
plt.tight_layout()
plt.savefig('road_context_2_side_tagging.png', dpi=100)
plt.show()

print('Side tagging summary:')
for side in model['sides']:
    adj = side.get('adjacent_road')
    if adj:
        print(f"  side {side['edge_index']} ({side['cardinal_hint']}): "
              f"{adj['name']} | {adj['hierarchy']} | {adj['width_m']} m wide "
              f"| {adj['distance_m']} m away | {adj['frontage_m']:.1f} m frontage")
    else:
        print(f"  side {side['edge_index']} ({side['cardinal_hint']}): no adjacent road")

In [ ]:
# ---------------------------------------------------------------------------
# §3 — Setback derivation: road widths → per-edge setbacks → buildable zone
# ---------------------------------------------------------------------------

erw = roads_result['edge_road_widths']
print('edge_road_widths (from real road objects):', erw)
print('Derived setbacks per edge (road_width × 0.4, min 3 m):')
for i, w in sorted(erw.items()):
    sb = max(3.0, w * 0.4)
    print(f'  side {i}: road {w} m → setback {sb:.1f} m')

# Buildable zone with road-derived setbacks
buildable_with_roads = compute_buildable_zone(
    SITE, default_setback=5.0, edge_road_widths=erw
)
buildable_no_roads = compute_buildable_zone(SITE, default_setback=5.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, bz, title in [
    (axes[0], buildable_no_roads, 'Uniform 5 m setback (no road data)'),
    (axes[1], buildable_with_roads, 'Road-derived setbacks (main road = 8 m)'),
]:
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10, fontweight='bold')
    sp = [(p[0], p[1]) for p in SITE]
    xs, ys = zip(*sp)
    ax.fill(xs, ys, alpha=0.10, color='#457B9D')
    ax.plot(xs, ys, '-', color='#1D3557', lw=1.5, label='Site')
    if not bz.is_empty:
        bzx, bzy = bz.exterior.xy
        ax.fill(bzx, bzy, alpha=0.35, color='#2A9D8F', label='Buildable zone')
        ax.plot(bzx, bzy, '-', color='#2A9D8F', lw=1.5)
        ax.text(0.5, 0.03,
                f'Buildable area: {bz.area:.0f} m²',
                ha='center', transform=ax.transAxes, fontsize=9,
                color='#2A9D8F')
    # Roads
    for road in [MAIN_ROAD, SECONDARY_ROAD, PATH_ROAD]:
        cl = road['centerline']
        ax.plot([p[0] for p in cl], [p[1] for p in cl],
                '--', color=HIER_COLOUR[road['hierarchy']], lw=1.5, alpha=0.6)
    ax.legend(fontsize=8)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')

plt.suptitle('§3  Road-derived setbacks create a larger buffer on the main-road side',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('road_context_3_setbacks.png', dpi=100)
plt.show()

print(f'\nBuildable area WITHOUT road setbacks: {buildable_no_roads.area:.0f} m²')
print(f'Buildable area WITH road setbacks:    {buildable_with_roads.area:.0f} m²')
print(f'Additional setback area (road margin): {buildable_no_roads.area - buildable_with_roads.area:.0f} m²')

In [ ]:
# ---------------------------------------------------------------------------
# §4 — Grid integration: main-road side drives the alignment axis
# ---------------------------------------------------------------------------

# With roads: grid aligns to the main-road side automatically.
grid_with_roads = derive_site_grid(model, spacing=12.0)

# Without roads: grid defaults to the longest side.
model_no_roads = build_site_model(SITE, {'default_setback': 5.0})
grid_no_roads = derive_site_grid(model_no_roads, spacing=12.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, grid, title, model_used in [
    (axes[0], grid_no_roads, 'No roads — longest-side fallback', model_no_roads),
    (axes[1], grid_with_roads, 'With roads — main-road side alignment', model),
]:
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10, fontweight='bold')
    sp = [(p[0], p[1]) for p in SITE]
    xs, ys = zip(*sp)
    ax.fill(xs, ys, alpha=0.10, color='#457B9D')
    ax.plot(xs, ys, '-', color='#1D3557', lw=1.5)

    # Grid lines
    for gl in grid.get('grid_lines', []):
        ax.plot([gl[0][0], gl[1][0]], [gl[0][1], gl[1][1]],
                '-', color='#95A3B3', lw=0.5, alpha=0.6)
    # Grid nodes
    gnx = [n[0] for n in grid.get('grid_nodes', [])]
    gny = [n[1] for n in grid.get('grid_nodes', [])]
    ax.scatter(gnx, gny, s=8, color='#264653', zorder=3, alpha=0.7)

    # Alignment side
    asi = grid['alignment_side_index']
    sides_used = model_used['sides']
    corners_used = model_used['corners']
    s = sides_used[asi]
    pA = corners_used[s['from_node_index']]['point']
    pB = corners_used[s['to_node_index']]['point']
    ax.plot([pA[0], pB[0]], [pA[1], pB[1]], '-', color='#E63946', lw=4, alpha=0.7,
            label=f"Alignment side {asi} ({grid.get('alignment_side_label','')})",
            zorder=4)

    # Main road (if present)
    if model_used.get('roads', {}).get('available'):
        cl = model_used['roads']['main_road']['centerline']
        ax.plot([p[0] for p in cl], [p[1] for p in cl],
                '-', color='#E63946', lw=2.5, ls='--', alpha=0.6, label='Main road')

    ax.text(0.5, 0.03, f"Grid angle: {grid['angle_deg']:.1f}° | {grid['node_count']} nodes",
            ha='center', transform=ax.transAxes, fontsize=9)
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')

plt.suptitle('§4  Phase 2 feeds Phase 3: main-road side drives grid alignment',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('road_context_4_grid_alignment.png', dpi=100)
plt.show()

print(f'No-road grid aligns to side {grid_no_roads["alignment_side_index"]}',
      f'(longest side, angle {grid_no_roads["angle_deg"]:.1f}°)')
print(f'Road-aware grid aligns to side {grid_with_roads["alignment_side_index"]}',
      f'(main-road side, angle {grid_with_roads["angle_deg"]:.1f}°)')

In [ ]:
# ---------------------------------------------------------------------------
# §5 — Ambiguity path: no road data supplied
# ---------------------------------------------------------------------------

model_ambig = build_site_model(SITE, {'default_setback': 5.0})  # no site_objects
roads_ambig = model_ambig['roads']

print('=== Ambiguity path (no roads) ===')
print('available:', roads_ambig['available'])
print('ambiguity code:', roads_ambig['ambiguity'])
print('ambiguity message:', roads_ambig['ambiguity_message'])
print('main_road:', roads_ambig['main_road'])
print('main_road_side_index:', roads_ambig['main_road_side_index'])
print()
print('Grid falls back to longest-side default:')
grid_fb = derive_site_grid(model_ambig, spacing=12.0)
print(f'  alignment_side_index = {grid_fb["alignment_side_index"]}',
      f'(longest side, angle {grid_fb["angle_deg"]:.1f}°)')
print()
print('Summary: no road data → ambiguity recorded → grid uses longest-side fallback.')
print('The agent should surface this ambiguity to the user rather than inventing a road.')

In [ ]:
# ---------------------------------------------------------------------------
# §6 — Summary: per-road analysis table
# ---------------------------------------------------------------------------

print('=== Phase 2 Road Context — Summary ===' )
print(f'{"Road":<20} {"Hierarchy":<12} {"Width":>8} {"Nearest side":>14}',
      f'{"Distance":>10} {"Frontage":>10}')
print('-' * 80)
for r in roads_result['roads']:
    print(f"{r.get('name','?'):<20} {r['hierarchy']:<12}",
          f"{r['width_m']:>7.1f}m {r['nearest_side_index']:>13}",
          f"{r['distance_m']:>9.1f}m {r['frontage_m']:>9.1f}m")
print('-' * 80)
mr = roads_result['main_road']
print(f"Main road: {mr['name']} | hierarchy {mr['hierarchy']} | "
      f"width {mr['width_m']} m | fronts side {roads_result['main_road_side_index']}")
print(f"edge_road_widths: {roads_result['edge_road_widths']}")
print()
print('Phase 2 complete: road context is ready for Phase 3 grid alignment,',
      'Phase 4 parking (near-road allocation), and Phase 5 circulation (public entry).')